In [ ]:
!pip install faker
import pandas as pd
import numpy as np
from faker import Faker
import random
from datetime import datetime, timedelta

# Initialize Faker for realistic names and demographics
fake = Faker()
num_records = 10500  # Exceeding your 10,000+ record benchmark

data = []

for i in range(num_records):
    patient_id = f"PID-{1000 + i}"
    name = fake.name()
    age = random.randint(1, 90)
    gender = random.choice(['Male', 'Female', 'Other'])

    # Appointment & Discharge Logic
    app_date = fake.date_time_between(start_date='-2y', end_date='now')
    # Stay duration between 1 and 15 days
    stay_duration = random.randint(1, 15)
    discharge_date = app_date + timedelta(days=stay_duration, hours=random.randint(1, 23))

    department = random.choice(['Cardiology', 'Neurology', 'Pediatrics', 'General Medicine', 'ICU', 'Orthopedics'])
    billing_amount = round(random.uniform(500, 5000), 2)
    status = random.choice(['Discharged', 'Transferred', 'Under Treatment'])

    data.append([patient_id, name, age, gender, app_date, discharge_date, department, billing_amount, status])

# Create DataFrame
df = pd.DataFrame(data, columns=['Patient_ID', 'Name', 'Age', 'Gender', 'Appointment_Date',
                                 'Discharge_Date', 'Department', 'Billing_Amount', 'Status'])

# Save to CSV for Excel/Power BI use
df.to_csv('hospital_data.csv', index=False)
print(f"Successfully generated {num_records} records!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 35.9 MB/s eta 0:00:00
Successfully generated 10500 records!


In [ ]:
import pandas as pd

# 1. Load your generated data
df = pd.read_csv('hospital_data.csv')

# 2. Convert date columns to actual "Date" format
# This is crucial for 'Data Transformation' mentioned in your summary
df['Appointment_Date'] = pd.to_datetime(df['Appointment_Date'])
df['Discharge_Date'] = pd.to_datetime(df['Discharge_Date'])

print(f"Checking {len(df)} records for accuracy...")

# --- THE CLEANING PROCESS ---

# A. Check for Missing Values
missing_count = df.isnull().sum().sum()
df = df.dropna()

# B. Logic Check: Finding 'impossible' dates
invalid_mask = df['Discharge_Date'] < df['Appointment_Date']
logic_errors = df[invalid_mask]

# C. Fix Logic Errors to ensure 99% data accuracy
df_cleaned = df[~invalid_mask].copy()

# D. Create a new column: 'Stay_Duration'
# This helps track "Performance Metrics" listed in your experience
df_cleaned['Stay_Duration'] = (df_cleaned['Discharge_Date'] - df_cleaned['Appointment_Date']).dt.days

# --- FINAL REPORT ---
print(f"Total Rows Processed: {len(df)}")
print(f"Missing Values Found: {missing_count}")
print(f"Logic Errors Found & Removed: {len(logic_errors)}")
print(f"Final Cleaned Dataset Size: {len(df_cleaned)}")

# Save this "Master Clean" file
df_cleaned.to_csv('hospital_data_cleaned.csv', index=False)
print("\nSuccess! Your cleaned file 'hospital_data_cleaned.csv' is ready.")

Checking 10500 records for accuracy...
Total Rows Processed: 10500
Missing Values Found: 0
Logic Errors Found & Removed: 0
Final Cleaned Dataset Size: 10500

Success! Your cleaned file 'hospital_data_cleaned.csv' is ready.


In [ ]:
import sqlite3
import pandas as pd

# 1. Load your cleaned data
df_cleaned = pd.read_csv('hospital_data_cleaned.csv')

# 2. Create a virtual SQL Database in memory
conn = sqlite3.connect('hospital.db')
cursor = conn.cursor()

# 3. Transfer the data to a SQL table named 'Records'
df_cleaned.to_sql('Records', conn, if_exists='replace', index=False)

print("--- SQL Analysis Started ---")

# --- QUERY 1: Performance by Department (KPI Tracking) ---
# This matches your CV skill: "tracking KPIs and performance metrics"
query1 = """
SELECT Department,
       COUNT(*) as Total_Patients,
       ROUND(AVG(Billing_Amount), 2) as Avg_Bill,
       ROUND(AVG(Stay_Duration), 2) as Avg_Stay_Days
FROM Records
GROUP BY Department
ORDER BY Total_Patients DESC;
"""
dept_stats = pd.read_sql(query1, conn)
print("\n1. Department Performance:")
print(dept_stats)

# --- QUERY 2: Revenue Trends (Data-driven decision making) ---
# This identifies which segments are most profitable
query2 = """
SELECT Gender,
       SUM(Billing_Amount) as Total_Revenue
FROM Records
GROUP BY Gender;
"""
revenue_stats = pd.read_sql(query2, conn)
print("\n2. Revenue by Gender Segment:")
print(revenue_stats)

# --- QUERY 3: High-Risk/High-Value Patients ---
# Finding patients staying longer than 10 days
query3 = """
SELECT Name, Age, Department, Stay_Duration
FROM Records
WHERE Stay_Duration > 10
LIMIT 5;
"""
long_stay_patients = pd.read_sql(query3, conn)
print("\n3. Sample of Long-Stay Patients (>10 Days):")
print(long_stay_patients)

conn.close()

--- SQL Analysis Started ---

1. Department Performance:
         Department  Total_Patients  Avg_Bill  Avg_Stay_Days
0        Pediatrics            1843   2733.02           7.99
1        Cardiology            1768   2670.72           7.86
2  General Medicine            1766   2764.77           8.05
3         Neurology            1721   2739.79           7.96
4               ICU            1717   2732.44           7.94
5       Orthopedics            1685   2759.77           8.16

2. Revenue by Gender Segment:
   Gender  Total_Revenue
0  Female     9601863.49
1    Male     9645060.02
2   Other     9451422.02

3. Sample of Long-Stay Patients (>10 Days):
                 Name  Age   Department  Stay_Duration
0       Oscar Johnson   64  Orthopedics             11
1      Jennifer Short   24   Cardiology             15
2  Daniel Blankenship   85   Pediatrics             15
3         Ashley Hall   90          ICU             12
4        Shawn Phelps   38   Pediatrics             15
